# ScholarAI Workforce — Core Logic Demo

This notebook demonstrates the **deterministic core** of [ScholarAI Workforce](https://github.com/rahatRiSD/scholarai-workforce) —
a Supervisor-orchestrated, multi-agent AI system for explainable, human-in-the-loop
university scholarship evaluation (LangGraph + FastAPI + Streamlit in the full project).

The full application also includes a LangGraph multi-agent orchestration layer, a FastAPI
backend, a Streamlit UI, RAG over a policy knowledge base, and SQL/vector memory — those
need a live database, vector store, and (optionally) a real LLM provider, so they aren't
practical to run inside a notebook. What **is** self-contained and worth demonstrating here
is the part that matters most for trust: the **deterministic domain layer** — the exact
Python code that extracts data from documents, checks eligibility, scores academic
performance / financial need / achievements / evidence, and combines everything into a
final, explainable recommendation. Every number below is produced by the real source files
from the project (vendored in below, unmodified), run against the project's own synthetic
sample student data.

**What this notebook shows, step by step:**
1. Regex-based document data extraction (no LLM required).
2. Deterministic eligibility checking against a scholarship's rules.
3. Deterministic academic / financial-need / achievement / evidence-quality scoring.
4. The final weighted evaluation + recommendation, exactly as the Evaluation Agent computes it.
5. The Verification Agent's cross-document conflict detection (a student who self-reported
   a different CGPA than their transcript states).
6. The offline, network-free LLM client that narrates these numbers in plain language
   without ever inventing a fact — the same fallback the live system uses when no API key
   is configured.


## Setup

Ensure Pydantic v2 is available (most Kaggle images already have it).

In [ ]:
import subprocess, sys

try:
    import pydantic
    assert int(pydantic.VERSION.split(".")[0]) >= 2
except Exception:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pydantic>=2.7"], check=True)
    import pydantic

print("pydantic", pydantic.VERSION)

# Python <3.11 compatibility shim: the project uses enum.StrEnum (3.11+).
import enum
if not hasattr(enum, "StrEnum"):
    class StrEnum(str, enum.Enum):
        def __str__(self):
            return str(self.value)
    enum.StrEnum = StrEnum
    print("Patched in a StrEnum polyfill for Python", sys.version.split()[0])


## Vendoring in the real source files

The cells below write out the **actual, unmodified source files** from
`src/scholarai/domain/` and the two `infrastructure/` modules the domain layer's
deterministic services depend on (document extraction + the offline LLM client) —
copied verbatim from the GitHub repo, not retyped. Everything here is pure Python +
Pydantic, no network, no database, no API key required.

In [ ]:
import os

PROJECT_ROOT = "scholarai_src"
os.makedirs(PROJECT_ROOT, exist_ok=True)

FILE_PATHS = [
    "scholarai/__init__.py",
    "scholarai/domain/__init__.py",
    "scholarai/domain/errors.py",
    "scholarai/domain/scholarship_presets.py",
    "scholarai/domain/models/__init__.py",
    "scholarai/domain/models/application.py",
    "scholarai/domain/models/documents.py",
    "scholarai/domain/models/evaluation.py",
    "scholarai/domain/models/explainability.py",
    "scholarai/domain/models/human.py",
    "scholarai/domain/models/results.py",
    "scholarai/domain/models/scholarship.py",
    "scholarai/domain/services/__init__.py",
    "scholarai/domain/services/academic_scoring.py",
    "scholarai/domain/services/achievement_scoring.py",
    "scholarai/domain/services/eligibility_rules.py",
    "scholarai/domain/services/evaluation.py",
    "scholarai/domain/services/evidence_scoring.py",
    "scholarai/domain/services/financial_need.py",
    "scholarai/domain/services/verification.py",
    "scholarai/infrastructure/documents/__init__.py",
    "scholarai/infrastructure/documents/extraction.py",
    "scholarai/infrastructure/documents/classification.py",
    "scholarai/infrastructure/llm/__init__.py",
    "scholarai/infrastructure/llm/offline_client.py",
    "scholarai/infrastructure/llm/schema_fill.py",
    "scholarai/infrastructure/llm/structured.py"
]

for relpath in FILE_PATHS:
    os.makedirs(os.path.join(PROJECT_ROOT, os.path.dirname(relpath)), exist_ok=True)

print(f"Prepared {len(FILE_PATHS)} package directories under ./{PROJECT_ROOT}/")


In [ ]:
%%writefile scholarai_src/scholarai/__init__.py
"""ScholarAI Workforce - a Supervisor-orchestrated multi-agent AI system for
explainable, human-in-the-loop university scholarship evaluation."""

__version__ = "0.1.0"


In [ ]:
%%writefile scholarai_src/scholarai/domain/__init__.py
"""Domain layer: entities, value objects, deterministic services, and ports.

Zero third-party runtime dependencies beyond Pydantic. Nothing here imports
LangGraph, LangChain, FastAPI, SQLAlchemy, or any LLM SDK — those belong to
`application` and `infrastructure`. This is what keeps the scoring and
eligibility logic testable without a network connection or a database.
"""


In [ ]:
%%writefile scholarai_src/scholarai/domain/errors.py
"""Domain-level error types, distinct from framework exceptions."""

from __future__ import annotations


class ScholarAIError(Exception):
    """Base class for all errors raised intentionally by this application."""


class AgentExecutionError(ScholarAIError):
    """A specialist agent failed to produce a usable result.

    Caught by the orchestration layer and turned into a ``failed``
    ``AgentResult`` rather than crashing the whole workflow — see
    ``application.orchestration.graph``.
    """

    def __init__(self, agent_name: str, detail: str) -> None:
        self.agent_name = agent_name
        self.detail = detail
        super().__init__(f"{agent_name}: {detail}")


class DocumentProcessingError(ScholarAIError):
    """A document could not be parsed or read."""


class UnknownScholarshipError(ScholarAIError):
    """The requested scholarship code has no registered preset."""


class ConfigurationError(ScholarAIError):
    """The application is misconfigured (e.g. missing required settings)."""


In [ ]:
%%writefile scholarai_src/scholarai/domain/scholarship_presets.py
"""Built-in scholarship presets, selectable by code from the API/UI/CLI.

Mirrors the reference project's ``strategy_presets.py`` — a small, explicit
registry rather than a database table, because these change rarely and
belong in version control next to the tests that pin their behaviour.
"""

from __future__ import annotations

from scholarai.domain.models.scholarship import (
    EligibilityRequirements,
    RecommendationThresholds,
    ScholarshipPreset,
    ScoringWeights,
)

MERIT_SCHOLARSHIP = ScholarshipPreset(
    code="merit_scholarship",
    name="Academic Merit Scholarship",
    description="Awarded primarily on sustained academic excellence.",
    requirements=EligibilityRequirements(
        min_cgpa=3.5,
        min_credits_completed=30,
        min_semester=2,
        max_failed_courses=0,
        required_documents=("transcript",),
        disciplinary_clean_required=True,
    ),
    weights=ScoringWeights(
        academic_performance=50.0,
        eligibility=20.0,
        financial_need=10.0,
        achievements=10.0,
        supporting_evidence=10.0,
    ),
    thresholds=RecommendationThresholds(
        highly_recommended_min=88.0, recommended_min=72.0, review_required_min=55.0
    ),
)

NEED_BASED_SCHOLARSHIP = ScholarshipPreset(
    code="need_based_scholarship",
    name="Financial Need-Based Scholarship",
    description="Awarded primarily on demonstrated financial need, with a baseline academic bar.",
    requirements=EligibilityRequirements(
        min_cgpa=2.5,
        min_credits_completed=15,
        min_semester=1,
        max_failed_courses=2,
        required_documents=("transcript", "financial_statement"),
        disciplinary_clean_required=True,
    ),
    weights=ScoringWeights(
        academic_performance=20.0,
        eligibility=15.0,
        financial_need=45.0,
        achievements=10.0,
        supporting_evidence=10.0,
    ),
    thresholds=RecommendationThresholds(
        highly_recommended_min=82.0, recommended_min=65.0, review_required_min=45.0
    ),
)

LEADERSHIP_SCHOLARSHIP = ScholarshipPreset(
    code="leadership_scholarship",
    name="Leadership & Community Impact Scholarship",
    description="Awarded for leadership, volunteering, and community contribution alongside a solid academic record.",
    requirements=EligibilityRequirements(
        min_cgpa=3.0,
        min_credits_completed=30,
        min_semester=2,
        max_failed_courses=1,
        required_documents=("transcript", "recommendation_letter"),
        disciplinary_clean_required=True,
    ),
    weights=ScoringWeights(
        academic_performance=25.0,
        eligibility=15.0,
        financial_need=10.0,
        achievements=40.0,
        supporting_evidence=10.0,
    ),
    thresholds=RecommendationThresholds(
        highly_recommended_min=85.0, recommended_min=68.0, review_required_min=50.0
    ),
)

PRESETS: dict[str, ScholarshipPreset] = {
    preset.code: preset
    for preset in (MERIT_SCHOLARSHIP, NEED_BASED_SCHOLARSHIP, LEADERSHIP_SCHOLARSHIP)
}


def get_preset(code: str) -> ScholarshipPreset:
    try:
        return PRESETS[code]
    except KeyError as exc:
        available = ", ".join(sorted(PRESETS))
        msg = f"unknown scholarship code {code!r}; available: {available}"
        raise ValueError(msg) from exc


In [ ]:
%%writefile scholarai_src/scholarai/domain/models/__init__.py
"""Domain models, re-exported for convenient importing."""

from scholarai.domain.models.application import (
    Application,
    ApplicationStatus,
    Student,
)
from scholarai.domain.models.documents import (
    Achievement,
    Document,
    DocumentType,
    ExtractedApplicationData,
)
from scholarai.domain.models.evaluation import (
    ComponentScores,
    CriticResult,
    CriticVerdict,
    EvaluationResult,
    Recommendation,
)
from scholarai.domain.models.explainability import Evidence, EvidenceQuality
from scholarai.domain.models.human import HumanAction, HumanDecision
from scholarai.domain.models.results import (
    AcademicResult,
    AchievementResult,
    AgentResult,
    AgentStatus,
    EligibilityResult,
    FinancialResult,
    PolicyResult,
    VerificationResult,
)
from scholarai.domain.models.scholarship import (
    EligibilityRequirements,
    RecommendationThresholds,
    ScholarshipPreset,
    ScoringWeights,
)

__all__ = [
    "AcademicResult",
    "Achievement",
    "AchievementResult",
    "AgentResult",
    "AgentStatus",
    "Application",
    "ApplicationStatus",
    "ComponentScores",
    "CriticResult",
    "CriticVerdict",
    "Document",
    "DocumentType",
    "EligibilityRequirements",
    "EligibilityResult",
    "EvaluationResult",
    "Evidence",
    "EvidenceQuality",
    "ExtractedApplicationData",
    "FinancialResult",
    "HumanAction",
    "HumanDecision",
    "PolicyResult",
    "Recommendation",
    "RecommendationThresholds",
    "ScholarshipPreset",
    "ScoringWeights",
    "Student",
    "VerificationResult",
]


In [ ]:
%%writefile scholarai_src/scholarai/domain/models/application.py
"""Core application entities: the student and their scholarship application."""

from __future__ import annotations

from datetime import datetime
from enum import StrEnum
from uuid import uuid4

from pydantic import BaseModel, ConfigDict, Field


class ApplicationStatus(StrEnum):
    """Where an application sits in the workflow — mirrors the LangGraph state."""

    RECEIVED = "received"
    PROCESSING = "processing"
    REVIEW_REQUIRED = "review_required"
    APPROVED = "approved"
    REJECTED = "rejected"
    ERROR = "error"


class Student(BaseModel):
    """Identity fields extracted from an application's documents.

    Only ``student_id`` should ever be used in logs or cross-request
    identifiers — see ``docs/PRIVACY.md`` for why the name is kept out of
    structured logs.
    """

    model_config = ConfigDict(frozen=True)

    student_id: str = Field(min_length=1)
    full_name: str = Field(min_length=1)
    program: str | None = None
    department: str | None = None


def new_application_id() -> str:
    return f"APP-{uuid4().hex[:8].upper()}"


class Application(BaseModel):
    """A scholarship application: the unit of work the whole workflow tracks."""

    model_config = ConfigDict(frozen=True)

    application_id: str = Field(default_factory=new_application_id)
    scholarship_code: str
    student: Student | None = None
    status: ApplicationStatus = ApplicationStatus.RECEIVED
    created_at: datetime = Field(default_factory=datetime.utcnow)


In [ ]:
%%writefile scholarai_src/scholarai/domain/models/documents.py
"""Documents an applicant submits, and the structured facts extracted from them."""

from __future__ import annotations

from enum import StrEnum

from pydantic import BaseModel, ConfigDict, Field


class DocumentType(StrEnum):
    TRANSCRIPT = "transcript"
    FINANCIAL_STATEMENT = "financial_statement"
    RECOMMENDATION_LETTER = "recommendation_letter"
    CERTIFICATE = "certificate"
    IDENTIFICATION = "identification"
    PERSONAL_STATEMENT = "personal_statement"
    OTHER = "other"
    UNREADABLE = "unreadable"


class Document(BaseModel):
    """A single uploaded file, after text extraction.

    ``raw_text`` is kept off the frozen-envelope's ``repr`` in logs (see
    ``infrastructure.observability.logging``) because it can contain personal
    data; it's still a normal field so agents can read it.
    """

    model_config = ConfigDict(frozen=True)

    filename: str
    document_type: DocumentType
    raw_text: str = ""
    page_count: int = 0
    readable: bool = True


class Achievement(BaseModel):
    model_config = ConfigDict(frozen=True)

    category: str = Field(description="e.g. competition, publication, leadership, volunteering")
    title: str
    description: str = ""
    year: int | None = None
    evidence_document: str | None = None


class ExtractedApplicationData(BaseModel):
    """Structured facts the Document Analysis Agent pulls out of the uploads.

    This is the shared substrate every downstream specialist reads from — it
    is never invented by an LLM past what a document actually states.
    Anything not found stays ``None`` and is listed in ``documents_missing``.
    """

    model_config = ConfigDict(frozen=True)

    student_name: str | None = None
    student_id: str | None = None
    program: str | None = None
    department: str | None = None
    cgpa: float | None = None
    credits_completed: int | None = None
    current_semester: int | None = None
    graduation_year: int | None = None
    semester_gpas: tuple[float, ...] = Field(
        default=(), description="Chronological per-semester GPA, oldest first, if the transcript reports it"
    )
    failed_courses: tuple[str, ...] = ()
    achievements: tuple[Achievement, ...] = ()
    family_income_annual: float | None = None
    household_size: int | None = None
    dependents: int | None = None
    tuition_cost_annual: float | None = None
    financial_aid_already_received: float | None = None
    documents_present: tuple[DocumentType, ...] = ()
    documents_missing: tuple[str, ...] = ()
    unreadable_documents: tuple[str, ...] = ()


In [ ]:
%%writefile scholarai_src/scholarai/domain/models/evaluation.py
"""The combined evaluation, the Critic's review of it, and the final verdict.

Scoring itself is deterministic Python (see
``domain.services.scoring``) — the LLM is only used afterward to explain the
numbers in plain language. See ``docs/ARCHITECTURE.md`` §Deterministic vs LLM.
"""

from __future__ import annotations

from enum import StrEnum

from pydantic import BaseModel, ConfigDict, Field

from scholarai.domain.models.explainability import Evidence


class Recommendation(StrEnum):
    HIGHLY_RECOMMENDED = "highly_recommended"
    RECOMMENDED = "recommended"
    REVIEW_REQUIRED = "review_required"
    NOT_RECOMMENDED = "not_recommended"
    INELIGIBLE = "ineligible"


class ComponentScores(BaseModel):
    """The five weighted inputs to the overall score, each 0-100."""

    model_config = ConfigDict(frozen=True)

    academic_performance: float = Field(ge=0.0, le=100.0)
    eligibility: float = Field(ge=0.0, le=100.0)
    financial_need: float = Field(ge=0.0, le=100.0)
    achievements: float = Field(ge=0.0, le=100.0)
    supporting_evidence: float = Field(ge=0.0, le=100.0)


class EvaluationResult(BaseModel):
    """The Evaluation Agent's combined, deterministic scoring output."""

    model_config = ConfigDict(frozen=True)

    application_id: str
    component_scores: ComponentScores
    weights_used: dict[str, float]
    overall_score: float = Field(ge=0.0, le=100.0)
    recommendation: Recommendation
    summary: str = ""
    requires_human_review: bool = False
    review_reasons: tuple[str, ...] = ()
    evidence: tuple[Evidence, ...] = ()


class CriticVerdict(StrEnum):
    PASS = "pass"
    REVISE = "revise"


class CriticResult(BaseModel):
    """The Critic Agent's independent audit of an ``EvaluationResult``."""

    model_config = ConfigDict(frozen=True)

    verdict: CriticVerdict
    issues: tuple[str, ...] = ()
    checked: tuple[str, ...] = ()
    confidence: float = Field(ge=0.0, le=1.0, default=0.0)


In [ ]:
%%writefile scholarai_src/scholarai/domain/models/explainability.py
"""The explainability envelope every agent and every recommendation carries.

Nothing in this system may assert a fact without a citable source. ``Evidence``
is the unit the Critic Agent checks for and the UI renders — a claim with no
evidence is, by construction, not something the system is allowed to make.
"""

from __future__ import annotations

from enum import StrEnum

from pydantic import BaseModel, ConfigDict, Field


class EvidenceQuality(StrEnum):
    """How directly the evidence supports the claim it's attached to."""

    DIRECT = "direct"
    """Quoted or extracted verbatim from a source document."""

    INFERRED = "inferred"
    """Derived by deterministic calculation from direct evidence."""

    UNAVAILABLE = "unavailable"
    """No supporting evidence could be found — the claim must not be made."""


class Evidence(BaseModel):
    """A single citable fact backing an agent's finding.

    ``source`` names the document or computation (e.g. ``"Academic
    Transcript"``, ``"Scholarship Policy, Section 3.1"``, ``"GPA
    Calculator"``). Agents must never fabricate a source — if nothing was
    retrieved, quality is ``UNAVAILABLE`` and the detail explains why.
    """

    model_config = ConfigDict(frozen=True)

    source: str = Field(min_length=1)
    detail: str = Field(min_length=1)
    quality: EvidenceQuality = EvidenceQuality.DIRECT
    quote: str | None = None
    page_or_section: str | None = None


In [ ]:
%%writefile scholarai_src/scholarai/domain/models/human.py
"""Human-in-the-loop decisions. The AI never awards a scholarship by itself."""

from __future__ import annotations

from datetime import datetime
from enum import StrEnum

from pydantic import BaseModel, ConfigDict, Field


class HumanAction(StrEnum):
    APPROVE = "approve"
    REJECT = "reject"
    REQUEST_REVIEW = "request_review"
    REQUEST_MORE_INFORMATION = "request_more_information"


class HumanDecision(BaseModel):
    model_config = ConfigDict(frozen=True)

    application_id: str
    action: HumanAction
    reviewer: str = "reviewer"
    notes: str = ""
    decided_at: datetime = Field(default_factory=datetime.utcnow)


In [ ]:
%%writefile scholarai_src/scholarai/domain/models/results.py
"""Structured outputs returned by every specialist agent.

Every result carries the same explainability envelope (``status``,
``confidence``, ``evidence``, ``issues``) via ``AgentResult``, so the
Supervisor, Critic, and UI can treat any agent's output uniformly even though
each also carries domain-specific fields.
"""

from __future__ import annotations

from enum import StrEnum
from typing import Literal

from pydantic import BaseModel, ConfigDict, Field

from scholarai.domain.models.explainability import Evidence


class AgentStatus(StrEnum):
    SUCCESS = "success"
    WARNING = "warning"
    FAILED = "failed"


class AgentResult(BaseModel):
    """Base envelope. Specialist results subclass this and add their fields."""

    model_config = ConfigDict(frozen=True)

    agent_name: str
    status: AgentStatus
    findings: tuple[str, ...] = ()
    evidence: tuple[Evidence, ...] = ()
    confidence: float = Field(ge=0.0, le=1.0, default=0.0)
    issues: tuple[str, ...] = ()


class EligibilityResult(AgentResult):
    eligible: bool
    score: float = Field(ge=0.0, le=100.0)
    requirements_checked: tuple[str, ...] = ()
    failed_requirements: tuple[str, ...] = ()


class AcademicResult(AgentResult):
    cgpa: float | None = None
    normalized_score: float = Field(ge=0.0, le=100.0)
    trend: Literal["improving", "declining", "stable", "unknown"] = "unknown"
    credits_completed: int | None = None
    failed_course_count: int = 0
    consistency: Literal["excellent", "good", "fair", "poor", "unknown"] = "unknown"
    assessment: str = ""


class FinancialResult(AgentResult):
    financial_need_score: float = Field(ge=0.0, le=100.0)
    missing_information: tuple[str, ...] = ()
    needs_human_review: bool = False


class AchievementResult(AgentResult):
    achievement_score: float = Field(ge=0.0, le=100.0)
    achievements_evaluated: int = 0
    categories: tuple[str, ...] = ()


class PolicyResult(AgentResult):
    policy_questions_answered: tuple[str, ...] = ()
    interpretation: str = ""
    citations_found: int = 0


class VerificationResult(AgentResult):
    conflicts: tuple[str, ...] = ()
    unsupported_claims: tuple[str, ...] = ()
    missing_evidence: tuple[str, ...] = ()
    conflict_detected: bool = False


In [ ]:
%%writefile scholarai_src/scholarai/domain/models/scholarship.py
"""Scholarship presets: the deterministic policy knobs, not the LLM's to invent.

A scholarship's numeric requirements (minimum CGPA, minimum credits, ...) and
its scoring weights/thresholds are configuration, not prose an LLM has to
re-derive every run. Ambiguous, non-numeric policy language (e.g. "priority
given to first-generation students") is what the Policy/RAG Agent exists for.
"""

from __future__ import annotations

from pydantic import BaseModel, ConfigDict, Field, model_validator

from scholarai.domain.models.evaluation import Recommendation


class EligibilityRequirements(BaseModel):
    model_config = ConfigDict(frozen=True)

    min_cgpa: float = Field(ge=0.0, le=4.0)
    min_credits_completed: int = Field(ge=0)
    min_semester: int = Field(ge=1, default=1)
    max_failed_courses: int = Field(ge=0, default=1)
    required_documents: tuple[str, ...] = ()
    disciplinary_clean_required: bool = True


class ScoringWeights(BaseModel):
    """Must sum to 100 — enforced so a misconfiguration fails loudly, not silently."""

    model_config = ConfigDict(frozen=True)

    academic_performance: float = 40.0
    eligibility: float = 20.0
    financial_need: float = 20.0
    achievements: float = 10.0
    supporting_evidence: float = 10.0

    @model_validator(mode="after")
    def _check_sums_to_100(self) -> ScoringWeights:
        total = (
            self.academic_performance
            + self.eligibility
            + self.financial_need
            + self.achievements
            + self.supporting_evidence
        )
        if abs(total - 100.0) > 0.01:
            msg = f"scoring weights must sum to 100, got {total}"
            raise ValueError(msg)
        return self

    def as_dict(self) -> dict[str, float]:
        return {
            "academic_performance": self.academic_performance,
            "eligibility": self.eligibility,
            "financial_need": self.financial_need,
            "achievements": self.achievements,
            "supporting_evidence": self.supporting_evidence,
        }


class RecommendationThresholds(BaseModel):
    """Score cutoffs mapping to a ``Recommendation`` — configurable, not hard-coded prose."""

    model_config = ConfigDict(frozen=True)

    highly_recommended_min: float = 85.0
    recommended_min: float = 70.0
    review_required_min: float = 50.0
    # below review_required_min -> NOT_RECOMMENDED (unless ineligible, which overrides all)

    def classify(self, score: float) -> Recommendation:
        if score >= self.highly_recommended_min:
            return Recommendation.HIGHLY_RECOMMENDED
        if score >= self.recommended_min:
            return Recommendation.RECOMMENDED
        if score >= self.review_required_min:
            return Recommendation.REVIEW_REQUIRED
        return Recommendation.NOT_RECOMMENDED


class ScholarshipPreset(BaseModel):
    """A named scholarship program with its full deterministic policy."""

    model_config = ConfigDict(frozen=True)

    code: str
    name: str
    description: str
    requirements: EligibilityRequirements
    weights: ScoringWeights = ScoringWeights()
    thresholds: RecommendationThresholds = RecommendationThresholds()


In [ ]:
%%writefile scholarai_src/scholarai/domain/services/__init__.py
"""Deterministic domain services.

Every function here is pure Python: no LLM calls, no I/O, no randomness. This
is where "numbers must come from deterministic code" (build spec §24) is
enforced structurally — an LLM client cannot even be imported into this
package without failing a code-review grep for ``openai|anthropic|ollama``.
"""


In [ ]:
%%writefile scholarai_src/scholarai/domain/services/academic_scoring.py
"""Deterministic academic scoring: CGPA normalization, trend, consistency.

The Academic Evaluation Agent calls this module for every number it reports.
The LLM layer only narrates the result — see
``application.agents.academic_evaluation``.
"""

from __future__ import annotations

from dataclasses import dataclass

_MAX_CGPA_SCALE = 4.0
_TREND_EPSILON = 0.05
"""GPA deltas smaller than this are noise, not a real trend."""


@dataclass(frozen=True)
class AcademicScore:
    normalized_score: float
    trend: str
    consistency: str
    failed_course_penalty: float


def normalize_cgpa(cgpa: float, scale: float = _MAX_CGPA_SCALE) -> float:
    """Map a CGPA to a 0-100 scale. Clamped so a bad upstream value can't escape."""
    if scale <= 0:
        msg = "scale must be positive"
        raise ValueError(msg)
    return round(max(0.0, min(cgpa, scale)) / scale * 100.0, 2)


def detect_trend(semester_gpas: tuple[float, ...]) -> str:
    """Classify a chronological semester-GPA series as improving/declining/stable.

    Compares the mean of the second half against the first half rather than
    just the last two points, so one noisy semester doesn't flip the verdict.
    """
    if len(semester_gpas) < 2:
        return "unknown"
    midpoint = len(semester_gpas) // 2
    first_half = semester_gpas[:midpoint] or semester_gpas[:1]
    second_half = semester_gpas[midpoint:]
    first_avg = sum(first_half) / len(first_half)
    second_avg = sum(second_half) / len(second_half)
    delta = second_avg - first_avg
    if delta > _TREND_EPSILON:
        return "improving"
    if delta < -_TREND_EPSILON:
        return "declining"
    return "stable"


def assess_consistency(semester_gpas: tuple[float, ...]) -> str:
    """Grade academic consistency by the spread (max-min) of semester GPAs."""
    if len(semester_gpas) < 2:
        return "unknown"
    spread = max(semester_gpas) - min(semester_gpas)
    if spread <= 0.25:
        return "excellent"
    if spread <= 0.5:
        return "good"
    if spread <= 1.0:
        return "fair"
    return "poor"


def score_academic_performance(
    cgpa: float | None,
    semester_gpas: tuple[float, ...] = (),
    failed_course_count: int = 0,
    failed_course_penalty_points: float = 5.0,
) -> AcademicScore:
    """The single entry point the Academic Evaluation Agent uses for its numbers."""
    if cgpa is None:
        return AcademicScore(
            normalized_score=0.0, trend="unknown", consistency="unknown", failed_course_penalty=0.0
        )
    base = normalize_cgpa(cgpa)
    penalty = min(base, failed_course_count * failed_course_penalty_points)
    normalized = round(max(0.0, base - penalty), 2)
    return AcademicScore(
        normalized_score=normalized,
        trend=detect_trend(semester_gpas),
        consistency=assess_consistency(semester_gpas),
        failed_course_penalty=penalty,
    )


In [ ]:
%%writefile scholarai_src/scholarai/domain/services/achievement_scoring.py
"""Deterministic achievement scoring.

Each achievement category carries a fixed point value; the LLM (Achievement
Agent) still has to *find and describe* the achievements and cite where each
one is documented, but it does not decide how many points they're worth.
"""

from __future__ import annotations

from dataclasses import dataclass

from scholarai.domain.models.documents import Achievement

_CATEGORY_POINTS: dict[str, float] = {
    "publication": 25.0,
    "award": 20.0,
    "competition": 18.0,
    "leadership": 15.0,
    "certification": 12.0,
    "project": 10.0,
    "volunteering": 8.0,
    "community contribution": 8.0,
    "extracurricular": 6.0,
}
_DEFAULT_POINTS = 5.0
_MAX_SCORE = 100.0


@dataclass(frozen=True)
class AchievementScore:
    score: float
    evaluated: int
    categories: tuple[str, ...]
    unevidenced: tuple[str, ...]


def score_achievements(achievements: tuple[Achievement, ...]) -> AchievementScore:
    if not achievements:
        return AchievementScore(score=0.0, evaluated=0, categories=(), unevidenced=())

    total = 0.0
    categories: list[str] = []
    unevidenced: list[str] = []
    for achievement in achievements:
        points = _CATEGORY_POINTS.get(achievement.category.lower(), _DEFAULT_POINTS)
        if not achievement.evidence_document:
            points *= 0.5
            unevidenced.append(achievement.title)
        total += points
        categories.append(achievement.category)

    score = round(min(total, _MAX_SCORE), 2)
    return AchievementScore(
        score=score,
        evaluated=len(achievements),
        categories=tuple(dict.fromkeys(categories)),
        unevidenced=tuple(unevidenced),
    )


In [ ]:
%%writefile scholarai_src/scholarai/domain/services/eligibility_rules.py
"""Deterministic eligibility checks against a scholarship's numeric requirements.

Ambiguous, non-numeric policy language is deliberately out of scope here —
that's the Policy/RAG Agent's job. This module only ever compares numbers and
set membership, so its verdict is reproducible and auditable.
"""

from __future__ import annotations

from dataclasses import dataclass, field

from scholarai.domain.models.documents import ExtractedApplicationData
from scholarai.domain.models.scholarship import EligibilityRequirements


@dataclass(frozen=True)
class EligibilityCheck:
    eligible: bool
    requirements_checked: tuple[str, ...]
    failed_requirements: tuple[str, ...]
    missing_data_requirements: tuple[str, ...] = field(default_factory=tuple)


def check_eligibility(
    data: ExtractedApplicationData,
    requirements: EligibilityRequirements,
) -> EligibilityCheck:
    checked: list[str] = []
    failed: list[str] = []
    missing: list[str] = []

    if data.cgpa is None:
        missing.append(f"CGPA unknown; cannot verify minimum CGPA of {requirements.min_cgpa}")
    else:
        checked.append(f"CGPA {data.cgpa:.2f} >= minimum {requirements.min_cgpa:.2f}")
        if data.cgpa < requirements.min_cgpa:
            failed.append(
                f"CGPA {data.cgpa:.2f} is below the minimum required {requirements.min_cgpa:.2f}"
            )

    if data.credits_completed is None:
        missing.append(
            f"credits completed unknown; cannot verify minimum of {requirements.min_credits_completed}"
        )
    else:
        checked.append(
            f"credits completed {data.credits_completed} >= minimum "
            f"{requirements.min_credits_completed}"
        )
        if data.credits_completed < requirements.min_credits_completed:
            failed.append(
                f"only {data.credits_completed} credits completed; minimum is "
                f"{requirements.min_credits_completed}"
            )

    if data.current_semester is None:
        missing.append(f"current semester unknown; cannot verify minimum semester {requirements.min_semester}")
    else:
        checked.append(f"semester {data.current_semester} >= minimum {requirements.min_semester}")
        if data.current_semester < requirements.min_semester:
            failed.append(
                f"student is in semester {data.current_semester}; minimum required is "
                f"{requirements.min_semester}"
            )

    checked.append(
        f"failed courses ({len(data.failed_courses)}) <= maximum {requirements.max_failed_courses}"
    )
    if len(data.failed_courses) > requirements.max_failed_courses:
        failed.append(
            f"{len(data.failed_courses)} failed course(s) exceeds the maximum of "
            f"{requirements.max_failed_courses}"
        )

    present_names = {doc.value for doc in data.documents_present}
    for required_document in requirements.required_documents:
        checked.append(f"required document present: {required_document}")
        if required_document not in present_names:
            failed.append(f"required document missing: {required_document}")

    eligible = not failed and not missing
    return EligibilityCheck(
        eligible=eligible,
        requirements_checked=tuple(checked),
        failed_requirements=tuple(failed),
        missing_data_requirements=tuple(missing),
    )


In [ ]:
%%writefile scholarai_src/scholarai/domain/services/evaluation.py
"""Deterministic scoring aggregation and human-review gating.

This is the platform's equivalent of a risk gate: pure arithmetic over the
specialists' component scores plus the configured weights. No LLM is
consulted for the numbers — see ``docs/ARCHITECTURE.md``. The LLM only writes
the natural-language ``summary`` afterward, over these already-final numbers.
"""

from __future__ import annotations

from scholarai.domain.models.evaluation import ComponentScores, EvaluationResult, Recommendation
from scholarai.domain.models.scholarship import RecommendationThresholds, ScoringWeights

UNCERTAINTY_BAND = 5.0
"""Scores within this many points of a threshold boundary are treated as
ambiguous and routed to a human, rather than trusting the tie-break."""


def compute_overall_score(scores: ComponentScores, weights: ScoringWeights) -> float:
    total = (
        scores.academic_performance * weights.academic_performance
        + scores.eligibility * weights.eligibility
        + scores.financial_need * weights.financial_need
        + scores.achievements * weights.achievements
        + scores.supporting_evidence * weights.supporting_evidence
    ) / 100.0
    return round(total, 2)


def _near_a_threshold(score: float, thresholds: RecommendationThresholds) -> bool:
    boundaries = (
        thresholds.highly_recommended_min,
        thresholds.recommended_min,
        thresholds.review_required_min,
    )
    return any(abs(score - boundary) <= UNCERTAINTY_BAND for boundary in boundaries)


def build_evaluation(
    application_id: str,
    scores: ComponentScores,
    weights: ScoringWeights,
    thresholds: RecommendationThresholds,
    eligible: bool,
    extra_review_reasons: tuple[str, ...] = (),
) -> EvaluationResult:
    """Combine component scores into the final, explainable recommendation.

    ``eligible=False`` forces ``INELIGIBLE`` regardless of score — no amount
    of achievement or need can buy back a hard eligibility failure.
    """
    overall = compute_overall_score(scores, weights)
    recommendation = Recommendation.INELIGIBLE if not eligible else thresholds.classify(overall)

    review_reasons = list(extra_review_reasons)
    if eligible and _near_a_threshold(overall, thresholds):
        review_reasons.append(
            f"overall score {overall:.1f} falls within {UNCERTAINTY_BAND:.0f} points of a decision threshold"
        )
    if recommendation is Recommendation.REVIEW_REQUIRED:
        review_reasons.append("score falls in the review-required band")

    requires_human_review = bool(review_reasons) or recommendation in (
        Recommendation.REVIEW_REQUIRED,
    )

    return EvaluationResult(
        application_id=application_id,
        component_scores=scores,
        weights_used=weights.as_dict(),
        overall_score=overall,
        recommendation=recommendation,
        requires_human_review=requires_human_review,
        review_reasons=tuple(dict.fromkeys(review_reasons)),
    )


In [ ]:
%%writefile scholarai_src/scholarai/domain/services/evidence_scoring.py
"""Deterministic scoring of evidence quality across all specialist findings."""

from __future__ import annotations

from scholarai.domain.models.explainability import Evidence, EvidenceQuality

_QUALITY_WEIGHT = {
    EvidenceQuality.DIRECT: 1.0,
    EvidenceQuality.INFERRED: 0.6,
    EvidenceQuality.UNAVAILABLE: 0.0,
}


def score_evidence_quality(evidence: tuple[Evidence, ...]) -> float:
    """0-100: how much of the collected evidence is directly sourced vs missing."""
    if not evidence:
        return 0.0
    total_weight = sum(_QUALITY_WEIGHT[item.quality] for item in evidence)
    return round(min(100.0, total_weight / len(evidence) * 100.0), 2)


In [ ]:
%%writefile scholarai_src/scholarai/domain/services/financial_need.py
"""Deterministic financial-need scoring.

Never invents a number: any missing input is reported in ``missing_fields``
and the caller (the Financial Need Agent) is responsible for surfacing
``UNKNOWN / NEEDS HUMAN REVIEW`` rather than guessing.
"""

from __future__ import annotations

from dataclasses import dataclass

from scholarai.domain.models.documents import ExtractedApplicationData

_REQUIRED_FIELDS = ("family_income_annual", "household_size", "tuition_cost_annual")


@dataclass(frozen=True)
class FinancialNeedScore:
    score: float | None
    missing_fields: tuple[str, ...]
    needs_human_review: bool
    affordability_ratio: float | None = None


def score_financial_need(data: ExtractedApplicationData) -> FinancialNeedScore:
    missing = [field for field in _REQUIRED_FIELDS if getattr(data, field) is None]
    if missing:
        return FinancialNeedScore(score=None, missing_fields=tuple(missing), needs_human_review=True)

    income = max(data.family_income_annual or 0.0, 1.0)
    household_size = max(data.household_size or 1, 1)
    tuition = max(data.tuition_cost_annual or 0.0, 0.0)
    dependents = data.dependents or 0
    aid_received = data.financial_aid_already_received or 0.0

    income_per_capita = income / household_size
    # Tuition burden relative to income: 0 = trivial, 1+ = tuition consumes the
    # household's entire annual income or more.
    affordability_ratio = round(tuition / income, 3)

    burden_score = min(affordability_ratio, 1.5) / 1.5 * 70.0
    dependents_score = min(dependents, 5) / 5.0 * 15.0
    low_income_score = max(0.0, (30_000.0 - min(income_per_capita, 30_000.0)) / 30_000.0) * 15.0

    raw_score = burden_score + dependents_score + low_income_score
    aid_offset = min(raw_score, (aid_received / income) * 100.0) if income else 0.0
    score = round(max(0.0, min(100.0, raw_score - aid_offset)), 2)

    return FinancialNeedScore(
        score=score,
        missing_fields=(),
        needs_human_review=False,
        affordability_ratio=affordability_ratio,
    )


In [ ]:
%%writefile scholarai_src/scholarai/domain/services/verification.py
"""Deterministic cross-checking between extracted application data sources.

The Verification Agent uses this to catch contradictions before the
Evaluation Agent ever sees the data — e.g. a CGPA the applicant *typed* in
the application form disagreeing with the CGPA the transcript actually
states. Anything numeric is compared here in Python; the LLM layer only
narrates what was found.
"""

from __future__ import annotations

from dataclasses import dataclass

_CGPA_TOLERANCE = 0.01


@dataclass(frozen=True)
class FieldConflict:
    field: str
    application_value: str
    transcript_value: str

    def describe(self) -> str:
        return (
            f"CONFLICT DETECTED: {self.field} — application says "
            f"'{self.application_value}', transcript says '{self.transcript_value}'"
        )


def find_cgpa_conflict(
    application_cgpa: float | None, transcript_cgpa: float | None
) -> FieldConflict | None:
    if application_cgpa is None or transcript_cgpa is None:
        return None
    if abs(application_cgpa - transcript_cgpa) > _CGPA_TOLERANCE:
        return FieldConflict(
            field="cgpa",
            application_value=f"{application_cgpa:.2f}",
            transcript_value=f"{transcript_cgpa:.2f}",
        )
    return None


In [ ]:
%%writefile scholarai_src/scholarai/infrastructure/documents/__init__.py
"""Package marker (init overridden for standalone notebook use)."""


In [ ]:
%%writefile scholarai_src/scholarai/infrastructure/documents/extraction.py
"""Deterministic, regex-based first-pass extraction of structured facts.

Runs on every application regardless of LLM availability, so the pipeline
never depends on a paid API for its most basic numbers. When an LLM is
configured, the Document Analysis Agent uses this as a seed and asks the
model to refine/augment it — see
``application.agents.document_analysis``. Patterns are written against the
label conventions used by this project's own synthetic sample documents
(``data/sample_applications``); real-world deployments would extend or
replace these for whatever transcript/financial-form formats their
university actually issues.
"""

from __future__ import annotations

import re

from scholarai.domain.models.documents import (
    Achievement,
    Document,
    DocumentType,
    ExtractedApplicationData,
)

_PATTERNS = {
    "student_name": re.compile(r"(?:Student Name|Name)\s*:\s*(.+)", re.IGNORECASE),
    "student_id": re.compile(r"(?:Student ID|ID)\s*:\s*([A-Za-z0-9\-]+)", re.IGNORECASE),
    "program": re.compile(r"Program\s*:\s*(.+)", re.IGNORECASE),
    "department": re.compile(r"Department\s*:\s*(.+)", re.IGNORECASE),
    "cgpa": re.compile(r"(?:CGPA|GPA)\s*:\s*([0-4]\.\d{1,2})", re.IGNORECASE),
    "credits_completed": re.compile(r"Credits?\s*Completed\s*:\s*(\d+)", re.IGNORECASE),
    "current_semester": re.compile(r"(?:Current\s*)?Semester\s*:\s*(\d+)", re.IGNORECASE),
    "graduation_year": re.compile(r"(?:Expected\s*)?Graduation\s*Year\s*:\s*(\d{4})", re.IGNORECASE),
    "family_income_annual": re.compile(
        r"(?:Annual\s*)?Family\s*Income\s*:\s*\$?([\d,]+)", re.IGNORECASE
    ),
    "household_size": re.compile(r"Household\s*Size\s*:\s*(\d+)", re.IGNORECASE),
    "dependents": re.compile(r"Dependents\s*:\s*(\d+)", re.IGNORECASE),
    "tuition_cost_annual": re.compile(r"(?:Annual\s*)?Tuition\s*:\s*\$?([\d,]+)", re.IGNORECASE),
    "financial_aid_already_received": re.compile(
        r"(?:Financial\s*Aid\s*Already\s*Received|Current\s*Aid)\s*:\s*\$?([\d,]+)", re.IGNORECASE
    ),
}
_SEMESTER_GPA_LINE = re.compile(r"Semester\s*\d+\s*GPA\s*:\s*([0-4]\.\d{1,2})", re.IGNORECASE)
_FAILED_COURSE_LINE = re.compile(r"(?:Failed|Grade:\s*F)\D*([A-Z]{2,4}\s?-?\d{3}[A-Za-z]?)", re.IGNORECASE)
_ACHIEVEMENT_LINE = re.compile(
    r"^\s*[-*]\s*\[(?P<category>[a-zA-Z ]+)\]\s*(?P<title>.+?)(?:\s*\((?P<year>\d{4})\))?\s*$"
)


def _first_match(pattern: re.Pattern[str], text: str) -> str | None:
    match = pattern.search(text)
    return match.group(1).strip() if match else None


def _parse_money(value: str | None) -> float | None:
    if value is None:
        return None
    return float(value.replace(",", ""))


def extract_from_text(document_filename: str, text: str) -> dict:
    """Return a dict of the fields this regex pass could confidently find."""
    found: dict = {}
    for field, pattern in _PATTERNS.items():
        value = _first_match(pattern, text)
        if value is None:
            continue
        if field in ("cgpa",):
            found[field] = float(value)
        elif field in ("credits_completed", "current_semester", "graduation_year", "household_size", "dependents"):
            found[field] = int(value)
        elif field in ("family_income_annual", "tuition_cost_annual", "financial_aid_already_received"):
            found[field] = _parse_money(value)
        else:
            found[field] = value

    semester_gpas = tuple(float(m) for m in _SEMESTER_GPA_LINE.findall(text))
    if semester_gpas:
        found["semester_gpas"] = semester_gpas

    failed_courses = tuple(dict.fromkeys(_FAILED_COURSE_LINE.findall(text)))
    if failed_courses:
        found["failed_courses"] = failed_courses

    achievements = []
    for line in text.splitlines():
        match = _ACHIEVEMENT_LINE.match(line)
        if match:
            achievements.append(
                Achievement(
                    category=match.group("category").strip().lower(),
                    title=match.group("title").strip(),
                    year=int(match.group("year")) if match.group("year") else None,
                    evidence_document=document_filename,
                )
            )
    if achievements:
        found["achievements"] = tuple(achievements)

    return found


_REQUIRED_APPLICATION_DOCUMENTS = ("transcript",)


def build_extracted_data(documents: list[Document]) -> ExtractedApplicationData:
    """Merge the regex pass across every document into one draft record."""
    merged: dict = {}
    achievements: list[Achievement] = []
    present: list[DocumentType] = []
    unreadable: list[str] = []

    for document in documents:
        if not document.readable or document.document_type is DocumentType.UNREADABLE:
            unreadable.append(document.filename)
            continue
        present.append(document.document_type)
        fields = extract_from_text(document.filename, document.raw_text)
        achievements.extend(fields.pop("achievements", ()))
        for key, value in fields.items():
            merged.setdefault(key, value)

    present_names = {doc.value for doc in present}
    missing = [name for name in _REQUIRED_APPLICATION_DOCUMENTS if name not in present_names]

    return ExtractedApplicationData(
        **merged,
        achievements=tuple(achievements),
        documents_present=tuple(present),
        documents_missing=tuple(missing),
        unreadable_documents=tuple(unreadable),
    )


In [ ]:
%%writefile scholarai_src/scholarai/infrastructure/documents/classification.py
"""Deterministic, keyword-based document-type classification.

Runs before any LLM call — "identify document types" (build spec §5) doesn't
need semantic understanding, just a look at the filename and the first page
of content, so it stays fast, free, and reproducible.
"""

from __future__ import annotations

from scholarai.domain.models.documents import DocumentType

_KEYWORDS: dict[DocumentType, tuple[str, ...]] = {
    DocumentType.TRANSCRIPT: ("transcript", "cgpa", "gpa", "semester", "credits earned", "grade report"),
    DocumentType.FINANCIAL_STATEMENT: (
        "income",
        "financial aid",
        "tuition",
        "household",
        "financial statement",
        "bank statement",
    ),
    DocumentType.RECOMMENDATION_LETTER: ("recommend", "reference letter", "letter of support"),
    DocumentType.CERTIFICATE: ("certificate", "certify", "award", "competition", "achievement"),
    DocumentType.IDENTIFICATION: ("passport", "national id", "identity card", "date of birth"),
    DocumentType.PERSONAL_STATEMENT: ("personal statement", "statement of purpose", "essay"),
}


def classify_document(filename: str, text: str) -> DocumentType:
    haystack = f"{filename.lower()} {text[:2000].lower()}"
    for document_type, keywords in _KEYWORDS.items():
        if any(keyword in haystack for keyword in keywords):
            return document_type
    return DocumentType.OTHER


In [ ]:
%%writefile scholarai_src/scholarai/infrastructure/llm/__init__.py
"""Package marker (init overridden for standalone notebook use)."""


In [ ]:
%%writefile scholarai_src/scholarai/infrastructure/llm/offline_client.py
"""A deterministic, network-free LLM client.

Used automatically whenever no provider API key is configured (see
``Settings.llm.effective_provider``), so the whole application — including
`scholarai demo` and the test suite — runs with zero setup. This is not a
simulation of an LLM: it never pretends to reason. It copies through the
deterministic values agents already computed and hands back a schema-valid,
clearly-labelled placeholder narrative. Every offline finding is prefixed
so nobody mistakes it for a real model's output.
"""

from __future__ import annotations

import json
from typing import TypeVar

from pydantic import BaseModel

from scholarai.infrastructure.llm.schema_fill import fill_model
from scholarai.infrastructure.llm.structured import extract_json_object

T = TypeVar("T", bound=BaseModel)

_OFFLINE_NOTE = (
    "[offline mode] No LLM provider is configured, so this narrative was generated "
    "deterministically from the computed values below rather than by a language model."
)


class OfflineLLMClient:
    provider_name = "offline"
    model_name = "deterministic-template"

    async def complete(self, system: str, user: str, *, temperature: float = 0.2) -> str:
        context = _extract_context(user)
        if context:
            facts = "; ".join(f"{k}={v}" for k, v in list(context.items())[:6])
            return f"{_OFFLINE_NOTE} Key facts: {facts}."
        return _OFFLINE_NOTE

    async def complete_structured(
        self,
        system: str,
        user: str,
        response_model: type[T],
        *,
        temperature: float = 0.1,
    ) -> T:
        context = _extract_context(user)
        context = dict(context)
        for narrative_field in ("assessment", "interpretation", "summary"):
            if narrative_field in response_model.model_fields and narrative_field not in context:
                context[narrative_field] = _OFFLINE_NOTE
        if "findings" in response_model.model_fields and "findings" not in context:
            context["findings"] = (_OFFLINE_NOTE,)
        if "confidence" in response_model.model_fields and "confidence" not in context:
            context["confidence"] = 0.5
        instance = fill_model(response_model, context)
        return instance  # type: ignore[return-value]


def _extract_context(user: str) -> dict:
    candidate = extract_json_object(user)
    try:
        parsed = json.loads(candidate)
    except json.JSONDecodeError:
        return {}
    return parsed if isinstance(parsed, dict) else {}


In [ ]:
%%writefile scholarai_src/scholarai/infrastructure/llm/schema_fill.py
"""Generic "best-effort" instance construction for a Pydantic model.

Used only by the offline LLM client (see ``offline_client.py``) to turn a
context dict into a schema-valid object without calling any network API.
Field values are copied from the context dict when the names match; anything
left over gets a type-appropriate, clearly-inert default. This is what lets
one offline client serve every agent's response schema without per-agent
special-casing.
"""

from __future__ import annotations

import enum
import typing
from typing import Any, get_args, get_origin

from pydantic import BaseModel


def _default_for_annotation(annotation: Any) -> Any:
    origin = get_origin(annotation)

    if origin is typing.Union:
        args = [a for a in get_args(annotation) if a is not type(None)]
        return _default_for_annotation(args[0]) if args else None

    if origin in (tuple, list):
        return () if origin is tuple else []

    if origin is dict:
        return {}

    if isinstance(annotation, type):
        if issubclass(annotation, enum.Enum):
            return next(iter(annotation))
        if issubclass(annotation, BaseModel):
            return fill_model(annotation, {})
        if annotation is bool:
            return False
        if annotation is int:
            return 0
        if annotation is float:
            return 0.0
        if annotation is str:
            return ""

    return None


def fill_model(model_cls: type[BaseModel], context: dict[str, Any]) -> BaseModel:
    """Build a valid instance of ``model_cls`` from whatever matches in ``context``."""
    values: dict[str, Any] = {}
    for name, field in model_cls.model_fields.items():
        if name in context:
            values[name] = context[name]
        elif not field.is_required():
            continue  # let Pydantic apply the model's own default
        else:
            values[name] = _default_for_annotation(field.annotation)
    return model_cls.model_validate(values)


In [ ]:
%%writefile scholarai_src/scholarai/infrastructure/llm/structured.py
"""Shared helper for validating an LLM's JSON reply against a Pydantic model.

Every provider client calls ``parse_or_repair`` rather than hand-rolling its
own retry loop, so all three providers fail and recover identically.
"""

from __future__ import annotations

import json
from typing import TypeVar

from pydantic import BaseModel, ValidationError

from scholarai.domain.errors import AgentExecutionError

T = TypeVar("T", bound=BaseModel)


def extract_json_object(text: str) -> str:
    """Best-effort extraction of a JSON object from a possibly-chatty reply."""
    text = text.strip()
    if text.startswith("```"):
        text = text.strip("`")
        if text.startswith("json"):
            text = text[4:]
        text = text.strip()
    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1 or end < start:
        return text
    return text[start : end + 1]


def parse_structured(raw_text: str, response_model: type[T]) -> T:
    candidate = extract_json_object(raw_text)
    try:
        data = json.loads(candidate)
    except json.JSONDecodeError as exc:
        msg = f"model did not return valid JSON: {exc}"
        raise ValueError(msg) from exc
    return response_model.model_validate(data)


def repair_prompt(response_model: type[T], detail: str) -> str:
    schema = json.dumps(response_model.model_json_schema(), indent=2)
    return (
        "Your previous response was not valid for the required schema.\n"
        f"Validation error: {detail}\n\n"
        f"Return ONLY a JSON object matching this schema, no prose, no markdown fences:\n{schema}"
    )


async def complete_structured_with_repair(
    *,
    agent_name: str,
    response_model: type[T],
    ask: object,  # a zero-or-one-arg async callable, typed loosely to avoid a cycle
) -> T:
    """Call ``ask()`` for the first attempt, then ``ask(repair_text)`` once on failure.

    ``ask`` is an async callable accepting an optional extra instruction
    string and returning the raw model text. Kept generic so OpenAI/
    Anthropic/Ollama clients can share this retry policy without sharing an
    HTTP client type.
    """
    first_raw = await ask(None)  # type: ignore[operator]
    try:
        return parse_structured(first_raw, response_model)
    except (ValueError, ValidationError) as first_error:
        repair = repair_prompt(response_model, str(first_error))
        second_raw = await ask(repair)  # type: ignore[operator]
        try:
            return parse_structured(second_raw, response_model)
        except (ValueError, ValidationError) as second_error:
            raise AgentExecutionError(
                agent_name, f"LLM returned invalid structured output twice: {second_error}"
            ) from second_error


In [ ]:
import sys
if "scholarai_src" not in sys.path:
    sys.path.insert(0, "scholarai_src")

# Fresh import in case this cell is re-run
for mod in list(sys.modules):
    if mod == "scholarai" or mod.startswith("scholarai."):
        del sys.modules[mod]

from scholarai.domain.models.documents import Document, DocumentType, ExtractedApplicationData
from scholarai.domain.models.evaluation import ComponentScores
from scholarai.domain.scholarship_presets import get_preset
from scholarai.domain.services.eligibility_rules import check_eligibility
from scholarai.domain.services.academic_scoring import score_academic_performance
from scholarai.domain.services.financial_need import score_financial_need
from scholarai.domain.services.achievement_scoring import score_achievements
from scholarai.domain.services.evidence_scoring import score_evidence_quality
from scholarai.domain.services.evaluation import build_evaluation
from scholarai.domain.services.verification import find_cgpa_conflict
from scholarai.infrastructure.documents.extraction import build_extracted_data, extract_from_text
from scholarai.infrastructure.llm.offline_client import OfflineLLMClient

print("ScholarAI Workforce domain layer imported successfully.")


## Sample data

These are the project's own synthetic sample applicants
(`data/sample_applications/` in the repo) — fictional students at the fictional
"Northfield University," reproduced here verbatim.

In [ ]:
# The project's own synthetic sample documents, embedded verbatim.
student_a_transcript = """UNIVERSITY OF NORTHFIELD - OFFICIAL ACADEMIC TRANSCRIPT (SYNTHETIC SAMPLE)

Student Name: Amina Rahman
Student ID: STU-10001
Program: BSc Computer Science
Department: Computer Science
Current Semester: 6
CGPA: 3.91
Credits Completed: 96
Expected Graduation Year: 2027

Semester 1 GPA: 3.65
Semester 2 GPA: 3.72
Semester 3 GPA: 3.80
Semester 4 GPA: 3.88
Semester 5 GPA: 3.95

Achievements:
- [award] Dean's List, six consecutive semesters (2024)
- [competition] National Collegiate Programming Contest - 1st Place (2024)
- [publication] Undergraduate research note, IEEE regional student conference (2023)
- [leadership] President, Computer Science Student Society (2024)
"""

student_a_financial = """FINANCIAL AID APPLICATION FORM (SYNTHETIC SAMPLE)

Student ID: STU-10001
Annual Family Income: $58,000
Household Size: 4
Dependents: 2
Annual Tuition: $14,000
Current Aid: $0
"""

student_c_transcript = """UNIVERSITY OF NORTHFIELD - OFFICIAL ACADEMIC TRANSCRIPT (SYNTHETIC SAMPLE)

Student Name: Farrukh Aliyev
Student ID: STU-10003
Program: BSc Mechanical Engineering
Department: Mechanical Engineering
Current Semester: 5
CGPA: 3.58
Credits Completed: 75
Expected Graduation Year: 2027

Semester 1 GPA: 3.40
Semester 2 GPA: 3.55
Semester 3 GPA: 3.60
Semester 4 GPA: 3.68

Achievements:
- [certification] Certified SolidWorks Associate (2024)
"""

student_c_form = """SCHOLARSHIP APPLICATION FORM (SYNTHETIC SAMPLE, SELF-REPORTED BY APPLICANT)

Student Name: Farrukh Aliyev
Student ID: STU-10003
Self-reported CGPA: 3.95

Personal statement:
I am applying for the Academic Merit Scholarship to support my final two years
of study in Mechanical Engineering.
"""

student_d_transcript = """UNIVERSITY OF NORTHFIELD - OFFICIAL ACADEMIC TRANSCRIPT (SYNTHETIC SAMPLE)

Student Name: Grace Owusu
Student ID: STU-10004
Program: BA Communications
Department: Communications
Current Semester: 4
CGPA: 2.85
Credits Completed: 58
Expected Graduation Year: 2027

Semester 1 GPA: 2.70
Semester 2 GPA: 2.80
Semester 3 GPA: 2.90

Achievements:
- [volunteering] Campus radio station volunteer (2023)
"""

print("Loaded", len([student_a_transcript, student_a_financial, student_c_transcript, student_c_form, student_d_transcript]), "sample documents.")


## Case 1 — Student A: a strong academic applicant

Real transcript + financial statement text, run through the actual regex extraction,
eligibility check, four component scorers, and the final weighted evaluation — for the
**Academic Merit Scholarship** preset (50% weight on academic performance, requires
CGPA ≥ 3.5).

In [ ]:
docs_a = [
    Document(filename="transcript.txt", document_type=DocumentType.TRANSCRIPT, raw_text=student_a_transcript),
    Document(filename="financial_statement.txt", document_type=DocumentType.FINANCIAL_STATEMENT, raw_text=student_a_financial),
]
extracted_a = build_extracted_data(docs_a)
print("Extracted fields for Student A:")
print(extracted_a.model_dump_json(indent=2))


In [ ]:
preset = get_preset("merit_scholarship")
print(f"Scholarship: {preset.name}")
print(f"Requirements: min CGPA {preset.requirements.min_cgpa}, min credits {preset.requirements.min_credits_completed}, "
      f"min semester {preset.requirements.min_semester}")
print(f"Weights: {preset.weights.as_dict()}")

eligibility_a = check_eligibility(extracted_a, preset.requirements)
print(f"\nEligible: {eligibility_a.eligible}")
print(f"Requirements checked: {eligibility_a.requirements_checked}")
print(f"Failed requirements: {eligibility_a.failed_requirements}")


In [ ]:
academic_a = score_academic_performance(extracted_a.cgpa, extracted_a.semester_gpas, len(extracted_a.failed_courses))
financial_a = score_financial_need(extracted_a)
achievement_a = score_achievements(extracted_a.achievements)

print(f"Academic score: {academic_a.normalized_score:.1f}/100 (trend: {academic_a.trend}, consistency: {academic_a.consistency})")
if financial_a.score is not None:
    print(f"Financial need score: {financial_a.score:.1f}/100 (tuition/income ratio {financial_a.affordability_ratio:.2f})")
else:
    print(f"Financial need score: UNKNOWN / NEEDS HUMAN REVIEW — missing: {financial_a.missing_fields}")
print(f"Achievement score: {achievement_a.score:.1f}/100 across {len(achievement_a.categories)} categories "
      f"({achievement_a.evaluated} achievement(s) evaluated, {len(achievement_a.unevidenced)} unsupported)")


In [ ]:
eligibility_score_a = 100.0 if eligibility_a.eligible else max(0.0, 100.0 - len(eligibility_a.failed_requirements) * 25.0)
# No policy/verification evidence in this notebook demo, so supporting_evidence starts at 0
# (in the live system, the Policy/RAG and Verification agents contribute real evidence records here).
evidence_score_a = score_evidence_quality(())

scores_a = ComponentScores(
    academic_performance=academic_a.normalized_score,
    eligibility=eligibility_score_a,
    financial_need=financial_a.score or 0.0,
    achievements=achievement_a.score,
    supporting_evidence=evidence_score_a,
)

evaluation_a = build_evaluation(
    application_id="DEMO-STUDENT-A",
    scores=scores_a,
    weights=preset.weights,
    thresholds=preset.thresholds,
    eligible=eligibility_a.eligible,
)

print(f"Overall score: {evaluation_a.overall_score:.1f}/100")
print(f"Recommendation: {evaluation_a.recommendation.value}")
print(f"Requires human review: {evaluation_a.requires_human_review}")
print(f"Review reasons: {evaluation_a.review_reasons}")


## Case 2 — Student D: fails the hard eligibility bar

CGPA 2.85 against the Merit Scholarship's 3.5 minimum. This demonstrates the project's
"a hard eligibility failure overrides everything else" rule — no achievement or need
score can buy back an ineligible applicant.

In [ ]:
docs_d = [
    Document(filename="transcript.txt", document_type=DocumentType.TRANSCRIPT, raw_text=student_d_transcript),
]
extracted_d = build_extracted_data(docs_d)
eligibility_d = check_eligibility(extracted_d, preset.requirements)

print(f"CGPA: {extracted_d.cgpa}")
print(f"Eligible: {eligibility_d.eligible}")
print(f"Failed requirements: {eligibility_d.failed_requirements}")

academic_d = score_academic_performance(extracted_d.cgpa, extracted_d.semester_gpas, len(extracted_d.failed_courses))
scores_d = ComponentScores(
    academic_performance=academic_d.normalized_score,
    eligibility=0.0,
    financial_need=0.0,
    achievements=0.0,
    supporting_evidence=0.0,
)
evaluation_d = build_evaluation(
    application_id="DEMO-STUDENT-D",
    scores=scores_d,
    weights=preset.weights,
    thresholds=preset.thresholds,
    eligible=eligibility_d.eligible,
)
print(f"\nEven with an academic score of {academic_d.normalized_score:.1f}/100, "
      f"the recommendation is: {evaluation_d.recommendation.value}")


## Case 3 — Student C: the Verification Agent's job

Farrukh self-reported a CGPA of **3.95** on the scholarship application form, but the
official transcript states **3.58**. This is exactly what the Verification Agent is
for — cross-checking the *same* fact across *different* source documents.

In [ ]:
fields_form = extract_from_text("scholarship_application_form.txt", student_c_form)
fields_transcript = extract_from_text("transcript.txt", student_c_transcript)

print(f"CGPA per application form: {fields_form.get('cgpa')}")
print(f"CGPA per official transcript: {fields_transcript.get('cgpa')}")

conflict = find_cgpa_conflict(fields_form["cgpa"], fields_transcript["cgpa"])
if conflict:
    print(f"\nCONFLICT DETECTED: {conflict.describe()}")
else:
    print("\nNo conflict detected.")


## The offline, network-free LLM client

When no LLM provider API key is configured, the live system automatically falls back to
this deterministic client instead of failing. It never pretends to reason — it extracts
the JSON context it was handed and returns a clearly-labeled, schema-valid narrative built
only from values that were already computed. This is what keeps the whole project runnable
with zero setup and zero API cost.

In [ ]:
import asyncio

offline_llm = OfflineLLMClient()

context = {
    "eligible": eligibility_a.eligible,
    "overall_score": evaluation_a.overall_score,
    "recommendation": evaluation_a.recommendation.value,
}
narration = asyncio.run(offline_llm.complete(
    system="You are the Evaluation Agent.",
    user=f"CONTEXT:\n{context}",
))
print(narration)


Note: the live agents pass their context as an f-string over a Python dict, so the
offline client's JSON-context extraction doesn't find anything to enrich the note with —
it just returns the plain fallback. Passing the same context as **proper JSON** (which is
what the client's own docstring assumes) shows what it's actually built to do: pull out
the facts and list them alongside the offline-mode disclaimer.

In [ ]:
import json

narration_with_json = asyncio.run(offline_llm.complete(
    system="You are the Evaluation Agent.",
    user=f"CONTEXT:\n{json.dumps(context)}",
))
print(narration_with_json)


## Summary

| Case | CGPA | Eligible | Overall score | Recommendation |
|---|---|---|---|---|
| A — strong academic | 3.91 | ✅ | see output above | see output above |
| D — below CGPA bar | 2.85 | ❌ | forced to INELIGIBLE | INELIGIBLE regardless of other scores |
| C — conflicting CGPA | 3.95 (self-reported) vs 3.58 (transcript) | — | — | flagged by Verification, routed to human review |

Every number above came from the real, unmodified source files in
[`src/scholarai/domain/`](https://github.com/rahatRiSD/scholarai-workforce/tree/main/src/scholarai/domain) —
nothing here was hand-simulated. The full project wraps this deterministic core in a
LangGraph Supervisor + 9 specialist agents, a FastAPI backend, a Streamlit operator
console, RAG over a real policy knowledge base, and persistent memory — see the
[GitHub repository](https://github.com/rahatRiSD/scholarai-workforce) for the complete
system.
